____
# Colocalization dataset
- put all variables for reconstruction in the same file


In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir, surface_drifters, depth_drifters, depth_100, depth_50
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

c0 ={'acc':'#941717', 'coriolis':'#388E3C','ggrad':'#42A5F5', 'wind':'#FFA000'}

/Users/mdemol/code/pynsitu/pynsitu/__init__.py:45: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  hour = Timedelta("1H")


_______
# Data

In [8]:
drifters_sources = 'all_med_variational_10min_v0.nc'
#DRIFTERS = {'all_med_variational_10min_v0' : os.path.join(zarr_dir, 'drifters_all_med_variational_10min_v0.csv')}
drifter_file = os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv'))
ALTI = {
    'swot250naive' : os.path.join(zarr_dir, 'swot_250_'+drifters_sources.replace('.nc', '')+'_naive.csv'), 
    'swot250gauss1e3' : os.path.join(zarr_dir, 'swot_250_'+drifters_sources.replace('.nc', '')+'_gauss1e3.csv'), 
}
WIND = {'era5' : os.path.join(zarr_dir, 'era5_'+drifters_sources.replace('.nc', '')+'.csv')}

In [9]:
def prepared_drifters() : 
    dfr = pd.read_csv(drifter_file).set_index('row_number')
    dfr['f'] =  2 * 2 * np.pi / 86164.1 * np.sin(dfr.latitude * np.pi / 180)
    dfr['coriolise']= -dfr.f * dfr.velocity_north
    dfr['coriolisn']= dfr.f * dfr.velocity_east
    dfr = dfr.rename(columns = {'acceleration_north':'accn', 'acceleration_east':'acce'})
    return dfr[['datetime', 'longitude', 'latitude', 'pass_number','time_to_swot','cycle_number','cycle_date','drifter_id','drifter_type','accn', 'acce', 'coriolise', 'coriolisn']]

def prepared_alti(alti_key, ggrad_var=['duacs_ssha_karin_2_filtered','cvl_mean_dynamic_topography_cnes_cls_22','cvl_ocean_tide_fes_2022']) :
    dfs = pd.read_csv(ALTI[alti_key]).set_index('row_number')
    dfs = dfs[['ggrade_'+v for v in ggrad_var]+['ggradn_'+v for v in ggrad_var]]
    dfs['ggrade_eta'] = dfs.ggrade_duacs_ssha_karin_2_filtered + dfs.ggrade_cvl_mean_dynamic_topography_cnes_cls_22+dfs.ggrade_cvl_ocean_tide_fes_2022
    dfs['ggradn_eta'] = dfs.ggradn_duacs_ssha_karin_2_filtered + dfs.ggradn_cvl_mean_dynamic_topography_cnes_cls_22+dfs.ggradn_cvl_ocean_tide_fes_2022
    return dfs.rename(columns ={v : v+'_' +alti_key for v in dfs})
    
def prepared_wind(wind_key): 
    dfw = pd.read_csv(WIND[wind_key]).set_index('row_number')
    dfw = dfw[[v for v in dfw if ('winde' in v or 'windn' in v)]]
    return dfw.rename(columns ={v : v+'_' +wind_key for v in dfw})

In [10]:
df = pd.concat([prepared_drifters(), prepared_alti('swot250naive'), prepared_alti('swot250gauss1e3'), prepared_wind('era5')], axis=1)

#depth
depth = df['pass_number'].copy()
depth.loc[df['drifter_type'].isin(surface_drifters)]=0
depth.loc[df['drifter_type'].isin(depth_drifters)]=15
depth.loc[df['drifter_id'].isin(depth_50)]=50
depth.loc[df['drifter_id'].isin(depth_100)]=100

df['depth'] = depth

/var/folders/fn/z858c2qj1lz65xr0z5mdvbf40000gp/T/ipykernel_80280/2321505827.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dfr = pd.read_csv(drifter_file).set_index('row_number')


In [11]:
# select only nearest in time colocalisation
def select_nearest_swot_coloc(df):
    idx = df.groupby(['pass_number','cycle_number','drifter_id']).time_to_swot.idxmin()
    return df.loc[idx]

dfc = select_nearest_swot_coloc(df).dropna()
dfc

,datetime,longitude,latitude,pass_number,time_to_swot,cycle_number,cycle_date,drifter_id,drifter_type,accn,...,ggradn_duacs_ssha_karin_2_filtered_swot250gauss1e3,ggradn_cvl_mean_dynamic_topography_cnes_cls_22_swot250gauss1e3,ggradn_cvl_ocean_tide_fes_2022_swot250gauss1e3,ggrade_eta_swot250gauss1e3,ggradn_eta_swot250gauss1e3,winde0_era5,windn0_era5,winde15_era5,windn15_era5,depth
row_number,,,,,,,,,,,,,,,,,,,,,
214,2023-04-01 23:40:00,5.632249,42.255307,3,0 days 00:02:32.696536,478,2023-04-01 23:42:32.696536000,0-4351296,CARTHE,-2.339830e-05,...,-0.000003,-1.100597e-06,-4.378941e-07,-0.000017,-0.000005,-2.743812e-05,-1.816019e-05,-7.800848e-06,-2.609972e-06,0
1001,2023-04-01 23:40:00,5.558617,42.880708,3,0 days 00:02:32.696536,478,2023-04-01 23:42:32.696536000,0-4388557,CARTHE,-3.028962e-06,...,0.000001,2.512117e-05,-5.680131e-07,-0.000023,0.000026,-2.581363e-05,-1.853904e-05,-7.433077e-06,-2.806569e-06,0
500,2023-04-01 23:40:00,4.773443,42.198418,3,0 days 00:02:32.696536,478,2023-04-01 23:42:32.696536000,0-4388599,CARTHE,2.738240e-06,...,0.000008,9.421445e-07,-4.473702e-07,0.000015,0.000008,-3.597150e-05,-1.654748e-05,-9.757152e-06,-1.668382e-06,0
858,2023-04-01 23:40:00,5.600033,41.562868,3,0 days 00:02:32.696536,478,2023-04-01 23:42:32.696536000,2,MELODI,-1.010323e-05,...,0.000023,-2.018614e-06,-4.428720e-07,-0.000004,0.000020,-2.165497e-05,-1.082177e-05,-5.929494e-06,-1.212077e-06,0
357,2023-04-01 23:40:00,4.704619,41.945077,3,0 days 00:02:32.696536,478,2023-04-01 23:42:32.696536000,300534060211000,SVP,8.290957e-07,...,-0.000009,-3.127146e-06,-4.614135e-07,-0.000010,-0.000012,-3.471423e-05,-1.267564e-05,-9.203017e-06,-8.147569e-07,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
383048,2023-07-09 19:00:00,2.122807,41.093707,16,0 days 00:03:38.131544384,577,2023-07-09 18:56:21.868455616,295,HEREON,-9.669657e-06,...,0.000007,5.261937e-06,-7.280843e-07,0.000012,0.000011,9.494988e-08,6.632627e-07,6.584489e-08,1.540219e-07,0
382230,2023-07-09 22:40:00,1.552078,40.941677,16,0 days 03:43:38.131544384,577,2023-07-09 18:56:21.868455616,4694223,CARTHE,2.564552e-06,...,0.000008,2.828935e-06,-7.703162e-07,-0.000012,0.000010,5.884896e-07,1.531490e-06,2.412040e-07,3.317483e-07,0
382905,2023-07-09 19:00:00,1.794712,39.114167,16,0 days 00:03:38.131544384,577,2023-07-09 18:56:21.868455616,6204608,OSMC,1.810231e-05,...,-0.000022,-6.027253e-06,-1.372612e-06,0.000012,-0.000030,6.074151e-07,9.681990e-07,2.093266e-07,1.944995e-07,15


In [12]:
df.to_csv(os.path.join(zarr_dir, 'coloc_all.csv'))
dfc.to_csv(os.path.join(zarr_dir, 'coloc_nearest.csv'))

In [13]:
def find_combination(df):
    id_comb = []
    # Alti
    ggrade, ggradn = [v for v in df if 'ggrade' in v], [v for v in df if 'ggradn' in v]
    alti_var = [v.replace('ggrade_','') for v in ggrade]
    
    # Wind
    winde, windn = [v for v in df if 'winde' in v], [v for v in df if 'windn' in v]
    wind_var = [v.replace('winde','').replace('_', '') for v in winde]

    import itertools  

    couple = list(itertools.product(alti_var, wind_var))
    E, N = list(itertools.product(ggrade, winde)), list(itertools.product(ggradn, windn))
    
    id_comb = dict()
    for i in range(len(E)):
        le = {"acc" : "acce", "coriolis" : "coriolise", "ggrad":E[i][0], "wind":E[i][1]}
        ln = {"acc" : "accn", "coriolis" : "coriolisn", "ggrad":N[i][0], "wind":N[i][1]}
        id_comb['__'.join(couple[i])] = le, ln
    return id_comb

In [14]:
id_comb = find_combination(dfc)
id_comb

{'duacs_ssha_karin_2_filtered_swot250naive__0era5': ({'acc': 'acce',
   'coriolis': 'coriolise',
   'ggrad': 'ggrade_duacs_ssha_karin_2_filtered_swot250naive',
   'wind': 'winde0_era5'},
  {'acc': 'accn',
   'coriolis': 'coriolisn',
   'ggrad': 'ggradn_duacs_ssha_karin_2_filtered_swot250naive',
   'wind': 'windn0_era5'}),
 'duacs_ssha_karin_2_filtered_swot250naive__15era5': ({'acc': 'acce',
   'coriolis': 'coriolise',
   'ggrad': 'ggrade_duacs_ssha_karin_2_filtered_swot250naive',
   'wind': 'winde15_era5'},
  {'acc': 'accn',
   'coriolis': 'coriolisn',
   'ggrad': 'ggradn_duacs_ssha_karin_2_filtered_swot250naive',
   'wind': 'windn15_era5'}),
 'cvl_mean_dynamic_topography_cnes_cls_22_swot250naive__0era5': ({'acc': 'acce',
   'coriolis': 'coriolise',
   'ggrad': 'ggrade_cvl_mean_dynamic_topography_cnes_cls_22_swot250naive',
   'wind': 'winde0_era5'},
  {'acc': 'accn',
   'coriolis': 'coriolisn',
   'ggrad': 'ggradn_cvl_mean_dynamic_topography_cnes_cls_22_swot250naive',
   'wind': 'windn

In [15]:
def compute(df, id_comb):
    D = []
    for comb in id_comb :
        coords_list = ['datetime', 'longitude', 'latitude', 'pass_number','time_to_swot','cycle_number','drifter_id', 'drifter_type']
        ds = df[coords_list].to_xarray()
        import itertools
        ds['id_comb'] = comb
        
        for direction in ['e', 'n'] : 
            if direction == 'e' : l = id_comb[comb][0]
            if direction == 'n' : l = id_comb[comb][1]
                
            # sum 
            ds['sum'+direction] = sum([df[v] for v in l.values()])

            # simple var + sum- one term
            for v in list(l.keys()) : 
                ds[v+direction] = df[l[v]]
                ds['exc'+direction+'_'+v] = ds['sum'+direction]-ds[v+direction]
                
            #2 by 2 product    
            couple = list(itertools.combinations(list(l.keys()),2))
            for c in couple : 
                ds['prod'+direction+'_'+'_'.join(c)] = ds[c[0]+direction] * ds[c[1]+direction]
                
        D.append(ds)
    DS = xr.concat(D, dim='id_comb').set_coords(coords_list)
    return DS
            

In [16]:
DS = compute(dfc, id_comb)
DSS = DS.where(DS.drifter_type.isin(surface_drifters), drop=True)
DSD = DS.where(DS.drifter_type.isin(depth_drifters), drop=True)

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/core/dataset.py:4789: UserWarning: No index created for dimension id_comb because variable id_comb is not a coordinate. To create an index for id_comb, please first call `.set_coords('id_comb')` on this object.
  warnings.warn(


In [39]:
DS['drifter_id'] = DS.drifter_id.astype(str)

In [40]:
DS.to_netcdf(os.path.join(zarr_dir, 'coloc_closure.nc'))

In [41]:
DS

<xarray.Dataset> Size: 13MB
Dimensions:               (id_comb: 16, row_number: 2340)
Coordinates:
    datetime              (id_comb, row_number) object 300kB '2023-04-01 23:4...
    longitude             (id_comb, row_number) float64 300kB 5.632 ... 1.789
    latitude              (id_comb, row_number) float64 300kB 42.26 ... 39.33
    pass_number           (id_comb, row_number) int64 300kB 3 3 3 3 ... 16 16 16
    time_to_swot          (id_comb, row_number) object 300kB '0 days 00:02:32...
    cycle_number          (id_comb, row_number) int64 300kB 478 478 ... 577 577
    drifter_id            (id_comb, row_number) <U15 2MB '0-4351296' ... '300...
    drifter_type          (id_comb, row_number) object 300kB 'CARTHE' ... 'SVP'
  * id_comb               (id_comb) <U63 4kB 'duacs_ssha_karin_2_filtered_swo...
  * row_number            (row_number) int64 19kB 214 1001 500 ... 383191 382476
Data variables: (12/30)
    sume                  (id_comb, row_number) float64 300kB -1.606e-05 ... ...
    acce                  (id_comb, row_number) float64 300kB -1.517e-05 ... ...
    exce_acc              (id_comb, row_number) float64 300kB -8.948e-07 ... ...
    coriolise             (id_comb, row_number) float64 300kB 4.449e-05 ... 2...
    exce_coriolis         (id_comb, row_number) float64 300kB -6.055e-05 ... ...
    ggrade                (id_comb, row_number) float64 300kB -1.795e-05 ... ...
    ...                    ...
    prodn_acc_coriolis    (id_comb, row_number) float64 300kB -6.065e-10 ... ...
    prodn_acc_ggrad       (id_comb, row_number) float64 300kB 2.959e-10 ... -...
    prodn_acc_wind        (id_comb, row_number) float64 300kB 4.249e-10 ... 9...
    prodn_coriolis_ggrad  (id_comb, row_number) float64 300kB -3.277e-10 ... ...
    prodn_coriolis_wind   (id_comb, row_number) float64 300kB -4.707e-10 ... ...
    prodn_ggrad_wind      (id_comb, row_number) float64 300kB 2.296e-10 ... -...